# Analyse the Results of Running Moran Process Experiment on Different Graphs
This is the newest version of this analysis file, where I can merge the csv of different jobs. 

## Setup

Imports, configuration, and data loading. Set `BATCH_NAME` below to point at whichever batch you want to analyze.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process 

import sys
sys.path.insert(0, 'src')

import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
# import numpy as np
import seaborn as sns
import os
from pathlib import Path
from moran_process.analysis.analysis_utils import *
# change this if on a different computer!
from moran_process.core.population_graph import GRAPH_PROPS
# Set aesthetic parameters for "publication-quality" plots
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['lines.linewidth'] = 2.5


In [ ]:

BATCH_NAME = '2026-05-20_scaling_study_3'  # <-- change this to whatever batch you want to analyse
CATEGORY_FILTER = 'Random'  # set to e.g. 'Random', 'Complete', 'Cycle' to filter; None = all categories
ANIMAL_CATEGORIES = ["Mammalian", "Fish", "Avian"]



In [ ]:
ROOT = Path(os.getcwd()) 

# Now define your paths relative to ROOT
BATCH_DIR = ROOT / "simulation_data" / BATCH_NAME
FIGURES_DIR = BATCH_DIR / "figures"


In [ ]:
import glob

output_file = os.path.join(BATCH_DIR, f"temp_full_results.csv") 
tmp_results_path = os.path.join(BATCH_DIR, "tmp", "results")
all_files = glob.glob(os.path.join(tmp_results_path, "result_job_*.csv"))
print(f"Found {len(all_files)} files in temp results directory: {tmp_results_path}.")


### The Graph Features we analyse: 


In [ ]:
GRAPH_PROPERTY_DESCRIPTION.keys()

In [ ]:
GRAPH_PROPERTY_DESCRIPTION
for prop, desc in GRAPH_PROPERTY_DESCRIPTION.items():
    print(f"{prop.title().replace('_', ' '):40}: {desc}")

In [ ]:
results_df_path = aggregate_results_no_load(batch_dir=BATCH_DIR, delete_temp=False)

In [ ]:

lazy_df = pl.scan_csv(results_df_path)
schema = lazy_df.collect_schema()
n_rows = lazy_df.select(pl.len()).collect()[0, 0]
print("columns:", list(schema.keys()))
print(f"Raw data: {n_rows:,} rows x {len(schema)} columns")


In [ ]:
# --- Aggregating the Raw Simulations Result ---

df_graphs = pd.read_csv(os.path.join(BATCH_DIR, 'graph_props.csv'))
graph_statistics_path = BATCH_DIR / "graph_statistics.csv"
if not graph_statistics_path.exists:

    agg_results_df = (
        lazy_df
        .with_columns(
            pl.when(pl.col('fixation')).then(pl.col('steps')).otherwise(None).alias('steps_success')
        )
        .group_by(['wl_hash', 'r', 'graph_name'])
        .agg([
            pl.col('fixation').mean().alias('prob_fixation'),
            pl.col('steps_success').median().alias('median_steps'),
            pl.col('steps_success').mean().alias('mean_steps'),
            pl.col('steps_success').std().alias('std_steps'),
            pl.col('steps_success').quantile(0.25).alias('q25_steps'),
            pl.col('steps_success').quantile(0.75).alias('q75_steps'),
            (pl.col('steps_success').quantile(0.75) - pl.col('steps_success').quantile(0.25)).alias('iqr_steps'),
            pl.col('fixation').count().alias('n_grouped'),
        ])
        .collect(engine='streaming')  # constant-memory aggregation, safe for large batches
        .to_pandas()
    )

    print("Shape before merging: ", agg_results_df.shape)

    analysis_df = pd.merge(
        agg_results_df,
        df_graphs,
        on=['wl_hash', 'graph_name'],
        how='left',
        suffixes=('', '_db')
    )
    analysis_df['z_order'] = (analysis_df['category'] != 'Random').astype(int)
    analysis_df = analysis_df.sort_values('z_order').drop(columns='z_order')
    analysis_df.to_csv(os.path.join(BATCH_DIR, 'graph_statistics.csv'), index=False)
else: 
    print(f"Aggregated Simulation Graph Statistics already exists! Loading {graph_statistics_path}...")
    analysis_df = pd.read_csv(graph_statistics_path)
print("Shape after merging: ", analysis_df.shape)

if CATEGORY_FILTER is not None:
    analysis_df = analysis_df[analysis_df['category'] == CATEGORY_FILTER].copy()
    print(f"Filtered to '{CATEGORY_FILTER}': {len(analysis_df):,} rows")

print(f"Graph aggregated simuation statistics table columns: {analysis_df.columns}")
analysis_df.tail(20)


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(analysis_df['mean_steps'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Mean Steps')
plt.ylabel('Frequency')
plt.title('Distribution of Mean Steps of All Graphs ')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(
    analysis_df.loc[analysis_df['category'] == 'Random', 'mean_steps'].dropna(),
    bins=50,
    edgecolor='black',
    alpha=0.7
)
plt.xlabel('Mean Steps')
plt.ylabel('Frequency')
plt.title('Distribution of Mean Steps of Random Graphs ')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
categories = sorted(analysis_df['category'].dropna().unique().tolist())
category_color_dict = generate_robust_color_dict(analysis_df, COLOR_DICT)

print("Color dictionary:")
for cat, color in category_color_dict.items():
    print(f"  {cat:15}: {color}")

In [ ]:
# --- Violin Plots: same data, one violin per category ---
violin_fig_path = FIGURES_DIR / "violin_steps_by_category.png"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not try_load_cached(violin_fig_path):
    # Load only the columns needed for plotting (avoids reloading the full dataset)
    merged_raw = (
        pl.scan_csv(results_df_path)
        .select(['wl_hash', 'steps', 'fixation'])
        .with_columns(
            pl.when(pl.col('fixation')).then(pl.col('steps')).otherwise(None).alias('steps_success')
        )
        .join(
            pl.from_pandas(df_graphs[['wl_hash', 'category']]).lazy(),
            on='wl_hash',
            how='left'
        )
        .collect()
        .to_pandas()
    )

    palette = {cat: category_color_dict[cat] for cat in categories}

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.violinplot(
        data=merged_raw,
        x="category",
        y="steps_success",
        order=categories,
        palette=palette,
        inner="box",
        linewidth=1.2,
        ax=ax,
    )
    ax.set_xlabel("Category", fontsize=13)
    ax.set_ylabel("Steps to Fixation", fontsize=13)
    ax.set_title("Distribution of Steps to Fixation by Category", fontsize=14)
    plt.tight_layout()
    plt.savefig(violin_fig_path, bbox_inches='tight', dpi=150)
    print(f"[cache] Saved: {violin_fig_path.name}")
    plt.show()

In [ ]:
plot_two_property_effect(df=analysis_df, 
                         x_prop='n_nodes', 
                         y_prop='n_edges', 
                         outcome='mean_steps', 
                         color_dict=category_color_dict, 
                         highlight_categories=ANIMAL_CATEGORIES, 
                         figures_dir=FIGURES_DIR)

In [ ]:
plot_two_property_effect_hexbin(df=analysis_df, x_prop='n_nodes', y_prop='n_edges', outcome='mean_steps', color_dict=category_color_dict, 
                                highlight_categories=ANIMAL_CATEGORIES, figures_dir=FIGURES_DIR)


In [ ]:
plot_two_property_effect_hexbin(df=analysis_df, x_prop='density', y_prop='n_nodes', outcome='mean_steps', 
                                color_dict=category_color_dict, highlight_categories=ANIMAL_CATEGORIES, 
                                figures_dir=FIGURES_DIR)



In [ ]:
plot_hybrid_density(analysis_df, 
                    'density', 
                    'mean_steps', 
                    with_violin=False, 
                    color_dict=category_color_dict, 
                    highlight_categories=ANIMAL_CATEGORIES,
                    figures_dir=FIGURES_DIR)


In [ ]:
plot_hybrid_density(analysis_df, 
                    'transitivity', 
                    'mean_steps', 
                    with_violin=False, 
                    color_dict=category_color_dict, 
                    highlight_categories=ANIMAL_CATEGORIES,
                    figures_dir=FIGURES_DIR)
plot_two_property_effect_hexbin(df=analysis_df, x_prop='transitivity', y_prop='n_nodes', outcome='mean_steps', 
                                color_dict=category_color_dict, highlight_categories=ANIMAL_CATEGORIES, figures_dir=FIGURES_DIR)




In [ ]:
plot_hybrid_density(analysis_df, 'max_degree', 'prob_fixation', with_violin=False, color_dict=category_color_dict, 
                    highlight_categories=ANIMAL_CATEGORIES, size_property=None, figures_dir=FIGURES_DIR)


In [ ]:
df_to_plot = analysis_df[analysis_df['r'] == 1.1]

plot_hybrid_density(analysis_df, 'prob_fixation', 'mean_steps', with_violin=False, color_dict=category_color_dict, highlight_categories=ANIMAL_CATEGORIES, figures_dir=FIGURES_DIR)
plot_hybrid_density(analysis_df, 'mean_steps', 'prob_fixation', with_violin=False, color_dict=category_color_dict, highlight_categories=ANIMAL_CATEGORIES, figures_dir=FIGURES_DIR)

## Per-Property Effect on Fixation Time (`r = 1.1`)

Each plot below sweeps one graph structural property against mean fixation time under weak selection (`r = 1.1`). Animal graphs are highlighted; random graphs form the background distribution.  Look for properties where the animal graphs sit outside the random cloud -- those are structural features that may explain amplifier or suppressor behavior.

In [ ]:

# --- EXAMPLES OF USAGE ---
for prop in GRAPH_PROPS:
    plot_hybrid_density(df_to_plot, prop, 'mean_steps', density_threshold=50, with_violin=True, color_dict=category_color_dict, highlight_categories=ANIMAL_CATEGORIES, figures_dir=FIGURES_DIR)


## Per-Property Effect on Fixation Probability (`r = 1.1`)

Same sweep as above but with fixation probability on the y-axis. Cross-reference with the fixation time plots: a true amplifier should show both elevated probability *and* shorter fixation time relative to the neutral drift baseline (`rho = 1/N`).

In [ ]:

# --- EXAMPLES OF USAGE ---
for prop in GRAPH_PROPS:
    plot_hybrid_density(df_to_plot, prop, 'prob_fixation', density_threshold=50, with_violin=True, color_dict=category_color_dict, highlight_categories=ANIMAL_CATEGORIES, figures_dir=FIGURES_DIR)


In [ ]:
plot_hybrid_density(df_to_plot, 'degree_std', 'mean_steps', density_threshold=50, with_violin=False, color_dict=category_color_dict, figures_dir=FIGURES_DIR)

plt.figure(figsize=(10, 8))
plt.hexbin(df_to_plot['degree_std'], df_to_plot['mean_steps'], gridsize=20, cmap='YlOrRd', mincnt=1)
plt.xlabel('degree_std')
plt.ylabel('mean_steps')
plt.colorbar(label='count')
plt.title('Hexbin plot: degree_std vs mean_steps')
plt.show()